# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Question:** which pages should be reviewed first? This is a yes/no question with an observed label (`is_decayed`, the same as in ML-03 and ML-07), used as a ranking: each page gets a decay score, and the top of the list is reviewed.

**Review population:** pages with at least 100 impressions in 90 days (21,758 of the 26,612 labeled pages). Below that, a page's 90-day total is so small that the label is often forced: a page with 1 impression in 90 days that is labeled at all must have lost it in the last 30 days, so it counts as decayed. A first run without this floor let Gradient Boosting reach a precision of 1.00 at the top 10 by finding such pages (shown in section 3). The floor matches the 100-impression floor of the Week-4 rule, applies to every method equally, and was added after this check, not before.

**Compared on the same pages with the same metric:**
- **Baseline:** the Week-4 rule (flag pages with 100 to 499 impressions that have not been updated for 91+ days), recomputed in this notebook.
- **Logistic Regression:** the readable first model; its coefficients can be read directly.
- **Gradient boosting** (`HistGradientBoostingClassifier`): a stronger model that captures interactions and handles missing values natively (30% of pages have no word count).

**Metric:** precision@K (K = 10, 50, 100, 500) next to the base rate of 0.514, with AUC as a secondary check. Precision@K matches the action: editors can only review a short list.

**Not used:** clustering (the question is not about grouping pages) and random forest (gradient boosting already covers non-linear models, and a third model would add complexity without a new question). Complexity is kept only if the comparison earns it.

**Inputs:** 34 features (23 numeric, 11 categorical). Not allowed: the label columns (`trend_direction`, `trend_pct`), every `*_last_30d` and `*_prev_30d` column, and the identifiers. `trend_pct` is the change in impressions between the two 30-day windows (verified in ML-03), so those windows would let a model recompute the label. The 90-day totals that remain still overlap those windows, so part of what a model finds is a decline that has already happened; this is a limit of a single snapshot.

**Reproducibility:** seed 42 everywhere; library versions are printed below.

In [10]:
!test -d repo || git clone --depth 1 https://github.com/flyrank-bih/flyrank-ml-internship-starter.git repo
!cp -r repo/data .

In [11]:
import numpy as np
import pandas as pd
import sklearn

SEED = 42
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Loaded:", df.shape)

# Same label as ML-03 and ML-07
df = df[df["trend_pct"].notna()].reset_index(drop=True)
df["is_decayed"] = ((df["trend_direction"] == "down") & (df["trend_pct"] <= -30)).astype(int)

# Review population: enough traffic to judge a decline (same 100-impression floor as the Week-4 rule)
MIN_IMPRESSIONS = 100
n_labeled = len(df)
df = df[df["impressions_90d"] >= MIN_IMPRESSIONS].reset_index(drop=True)

# Not allowed as features: label columns, label-window columns, identifiers
FORBIDDEN = ["trend_direction", "trend_pct", "is_decayed"] + \
            [c for c in df.columns if c.endswith(("_last_30d", "_prev_30d"))]
IDS = ["content_id", "client_id"]
FEATURES = [c for c in df.columns if c not in FORBIDDEN + IDS]

X = df[FEATURES].copy()
y = df["is_decayed"].values
groups = df["client_id"].values

NUMERIC = X.select_dtypes(include="number").columns.tolist()
CATEGORICAL = [c for c in FEATURES if c not in NUMERIC]
for c in CATEGORICAL:
    X[c] = X[c].astype("category")

print("Labeled pages:", f"{n_labeled:,}", "| kept (>= 100 impressions in 90 days):", f"{len(df):,}")
print("Clients:", len(set(groups)), "| base rate:", round(y.mean(), 3))
print("Features:", len(FEATURES), f"({len(NUMERIC)} numeric, {len(CATEGORICAL)} categorical)")
print("Categorical:", CATEGORICAL)
print("Features with missing values (%):")
print((X.isna().mean() * 100).round(1)[lambda s: s > 0].sort_values(ascending=False).to_string())
print("\nsklearn", sklearn.__version__, "| pandas", pd.__version__, "| seed", SEED)

Loaded: (30000, 44)
Labeled pages: 26,612 | kept (>= 100 impressions in 90 days): 21,758
Clients: 30 | base rate: 0.514
Features: 34 (23 numeric, 11 categorical)
Categorical: ['competition_level', 'content_type', 'main_intent', 'provider_used', 'model_used', 'age_tier', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'impression_tier', 'position_tier']
Features with missing values (%):
provider_used        71.8
word_count_tier      30.0
word_count           30.0
char_count           30.0
char_count_tier      30.0
model_used           24.4
competition_level     3.1
search_volume         2.6
cpc                   2.6
competition           2.6
main_intent           2.2
scroll_rate           0.1

sklearn 1.6.1 | pandas 2.2.3 | seed 42


## 2. Split design

**Design:** 5-fold cross-validation grouped by `client_id` (`GroupKFold`). Every client's pages sit entirely in the training side or entirely in the test side, never both. Each page is tested exactly once, and the model that scores it never saw its client.

**Why this is honest for the question:** the editorial queue will be used on pages from clients the model has not learned from, and pages from one client are alike (they share content style, history and traffic patterns). A random page-level split would put near-copies of a client's pages on both sides and inflate the scores.

**Same split for the baseline:** the Week-4 rule needs no training, so it is scored on the same test pages in every fold. Baseline and models are therefore judged on exactly the same rows.

**Not time-aware:** the starter data is a single snapshot, so a train-on-past, test-on-future split is not possible here. The label is a current-window proxy, not a future outcome; this limit is carried into the capstone.

**What to expect:** the folds are uneven. One client holds 26% of all pages and ends up alone in a test fold, and the base rate differs a lot between folds (table below). Results will therefore vary by fold, and fold-to-fold spread is reported next to the averages.

In [12]:
from sklearn.model_selection import GroupKFold

N_FOLDS = 5
FOLDS = list(GroupKFold(n_splits=N_FOLDS).split(df, y, groups))

rows, shared = [], 0
for i, (train_idx, test_idx) in enumerate(FOLDS, start=1):
    shared += len(set(groups[train_idx]) & set(groups[test_idx]))
    rows.append({"fold": i,
                 "train_pages": len(train_idx),
                 "test_pages": len(test_idx),
                 "test_clients": len(set(groups[test_idx])),
                 "test_base_rate": round(y[test_idx].mean(), 3)})

print(pd.DataFrame(rows).to_string(index=False))
tested = np.concatenate([test for _, test in FOLDS])
print("\nPages tested exactly once:", len(tested) == len(set(tested)) == len(df))
print("Clients shared between train and test (all folds):", shared)

 fold  train_pages  test_pages  test_clients  test_base_rate
    1        15191        6567             1           0.379
    2        17962        3796             6           0.533
    3        17959        3799             7           0.444
    4        17958        3800             8           0.717
    5        17962        3796             8           0.591

Pages tested exactly once: True
Clients shared between train and test (all folds): 0


## 3. Train + compare vs my baseline

Three methods are scored on the same 21,758 pages and the same five client-grouped folds from section 2: the Week-4 rule, Logistic Regression and Gradient boosting. Models are not tuned; their settings are fixed in the code (seed 42). The baseline needs no training, so it is scored on the same test pages.

**Table 1 (pooled):** every page is scored by a model that never saw its client, then the top K of all pages is taken, as the editorial queue would be.

**Table 2 (per fold):** the top 2% of each test fold, shown as lift over that fold's own base rate, with the spread across folds. Fold base rates differ a lot (section 2), so lift is the fair comparison there.

**Receipt:** the numbers are also written to `work/outputs/model_vs_baseline.json`.

**Why the floor:** the second cell repeats the Gradient boosting run without the 100-impression floor, to show what the floor removes.

In [13]:
import json, os
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# The folds from section 2 must belong to the current review population
assert sum(len(test) for _, test in FOLDS) == len(df), "FOLDS are stale: re-run the section 2 cell"

ids = df["content_id"].values
base_rate = y.mean()

def p_at_k(y_true, score, k, tie_ids):
    order = np.lexsort((tie_ids, -score))          # highest score first, ties broken by content_id
    return y_true[order[:k]].mean()

# Baseline: the Week-4 rule, same formula as ML-07 (no training)
flagged = ((df["impressions_90d"] >= 100) & (df["impressions_90d"] < 500)
           & (df["days_since_last_update"] >= 91))
stale_part = df["days_since_last_update"].clip(upper=365) / 365
volume_part = (500 - df["impressions_90d"]) / 400
baseline_score = np.where(flagged, stale_part * volume_part, 0.0)

# Logistic Regression: impute, scale numbers, one-hot the categories (read as text)
def make_logistic():
    prep = ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                          ("scale", StandardScaler())]), NUMERIC),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="missing")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore"))]), CATEGORICAL),
    ])
    return Pipeline([("prep", prep), ("model", LogisticRegression(max_iter=1000, random_state=SEED))])

X_lr = X.copy()
for c in CATEGORICAL:
    X_lr[c] = X_lr[c].astype(object)

# Gradient boosting: fixed settings (not tuned); handles missing values and categories natively
def make_boosting():
    return HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05,
                                          categorical_features="from_dtype", random_state=SEED)

# Out-of-fold scores: every page is scored by a model that never saw its client
oof = {"Logistic Regression": np.zeros(len(y)), "Gradient boosting": np.zeros(len(y))}
boosting_models = []
for train_idx, test_idx in FOLDS:
    lr = make_logistic().fit(X_lr.iloc[train_idx], y[train_idx])
    oof["Logistic Regression"][test_idx] = lr.predict_proba(X_lr.iloc[test_idx])[:, 1]
    gb = make_boosting().fit(X.iloc[train_idx], y[train_idx])
    oof["Gradient boosting"][test_idx] = gb.predict_proba(X.iloc[test_idx])[:, 1]
    boosting_models.append(gb)

scores = {"Baseline (Week-4 rule)": baseline_score, **oof}

# Table 1: pooled over all pages (top K of the whole review population)
Ks = [10, 50, 100, 500]
pooled = []
for name, s in scores.items():
    row = {"method": name}
    for k in Ks:
        row[f"P@{k}"] = round(p_at_k(y, s, k, ids), 3)
    row["lift@500"] = round(row["P@500"] / base_rate, 2)
    row["AUC"] = round(roc_auc_score(y, s), 3)
    pooled.append(row)
pooled_df = pd.DataFrame(pooled)
print(f"Base rate: {base_rate:.3f}  |  pages: {len(y):,}\n")
print("Pooled out-of-fold (top K of all pages):")
print(pooled_df.to_string(index=False))

# Table 2: per fold, top 2% of each test fold, as lift over that fold's own base rate
per_fold = {name: {"auc": [], "p": [], "lift": []} for name in scores}
for train_idx, test_idx in FOLDS:
    k = max(1, round(0.02 * len(test_idx)))
    fold_base = y[test_idx].mean()
    for name, s in scores.items():
        p = p_at_k(y[test_idx], s[test_idx], k, ids[test_idx])
        per_fold[name]["auc"].append(roc_auc_score(y[test_idx], s[test_idx]))
        per_fold[name]["p"].append(p)
        per_fold[name]["lift"].append(p / fold_base)

fold_rows = [{"method": name,
              "AUC": f"{np.mean(d['auc']):.3f} ± {np.std(d['auc']):.3f}",
              "P@top2%": f"{np.mean(d['p']):.3f} ± {np.std(d['p']):.3f}",
              "lift@top2%": f"{np.mean(d['lift']):.2f} ± {np.std(d['lift']):.2f}"}
             for name, d in per_fold.items()]
print("\nPer fold (5 folds, top 2% of each test fold, mean ± std):")
print(pd.DataFrame(fold_rows).to_string(index=False))

# Receipt: these numbers as a small JSON (no ids, no data)
os.makedirs("work/outputs", exist_ok=True)
receipt = {"seed": SEED, "review_population_pages": int(len(y)), "min_impressions_90d": MIN_IMPRESSIONS,
           "base_rate": round(float(base_rate), 3), "folds": len(FOLDS), "sklearn": sklearn.__version__,
           "pooled": pooled_df.to_dict(orient="records"), "per_fold_top2pct": fold_rows}
with open("work/outputs/model_vs_baseline.json", "w") as f:
    json.dump(receipt, f, indent=2)
print("\nWrote work/outputs/model_vs_baseline.json")

Base rate: 0.514  |  pages: 21,758

Pooled out-of-fold (top K of all pages):
                method  P@10  P@50  P@100  P@500  lift@500   AUC
Baseline (Week-4 rule)   0.5  0.68   0.64  0.618      1.20 0.511
   Logistic Regression   0.7  0.78   0.83  0.796      1.55 0.638
     Gradient boosting   0.5  0.76   0.72  0.710      1.38 0.667

Per fold (5 folds, top 2% of each test fold, mean ± std):
                method           AUC       P@top2%  lift@top2%
Baseline (Week-4 rule) 0.512 ± 0.013 0.604 ± 0.138 1.13 ± 0.11
   Logistic Regression 0.650 ± 0.017 0.820 ± 0.095 1.59 ± 0.26
     Gradient boosting 0.654 ± 0.026 0.759 ± 0.102 1.47 ± 0.32

Wrote work/outputs/model_vs_baseline.json


In [14]:
# The same Gradient boosting run WITHOUT the 100-impression floor
allp = pd.read_csv("data/raw/content_refresh_anonymized.csv")
allp = allp[allp["trend_pct"].notna()].reset_index(drop=True)
allp["is_decayed"] = ((allp["trend_direction"] == "down") & (allp["trend_pct"] <= -30)).astype(int)
X_all = allp[FEATURES].copy()
for c in CATEGORICAL:
    X_all[c] = X_all[c].astype("category")
y_all = allp["is_decayed"].values

oof_all = np.zeros(len(y_all))
for train_idx, test_idx in GroupKFold(n_splits=N_FOLDS).split(allp, y_all, allp["client_id"].values):
    m = make_boosting().fit(X_all.iloc[train_idx], y_all[train_idx])
    oof_all[test_idx] = m.predict_proba(X_all.iloc[test_idx])[:, 1]

tie = allp["content_id"].values
top100 = np.lexsort((tie, -oof_all))[:100]
print(f"Without the floor: {len(y_all):,} pages, base rate {y_all.mean():.3f}")
print(f"  P@10 {p_at_k(y_all, oof_all, 10, tie):.2f} | P@100 {p_at_k(y_all, oof_all, 100, tie):.2f} | "
      f"P@500 {p_at_k(y_all, oof_all, 500, tie):.3f} | AUC {roc_auc_score(y_all, oof_all):.3f}")
print(f"  Top 100 pages: median impressions_90d = {allp.loc[top100, 'impressions_90d'].median():.0f}, "
      f"share with 5 or fewer = {(allp.loc[top100, 'impressions_90d'] <= 5).mean():.0%}")

Without the floor: 26,612 pages, base rate 0.531
  P@10 1.00 | P@100 0.99 | P@500 0.866 | AUC 0.660
  Top 100 pages: median impressions_90d = 1, share with 5 or fewer = 97%


**Observed** (21,758 pages, base rate 0.514, client-grouped folds):

| Method | P@500 (pooled) | Lift@500 | AUC (pooled) | Per-fold lift, top 2% |
|---|---|---|---|---|
| Baseline (Week-4 rule) | 0.618 | 1.20x | 0.511 | 1.13 ± 0.11 |
| Logistic Regression | 0.796 | 1.55x | 0.638 | 1.59 ± 0.26 |
| Gradient boosting | 0.710 | 1.38x | 0.667 | 1.47 ± 0.32 |

- **Both models beat the baseline** at the top of the list. The baseline's overall ranking is close to chance (AUC 0.511): its precision comes from a narrow flagged group, not from ranking all pages.
- **The simpler model is not worse.** Logistic Regression is ahead on precision at the top (P@500 and per-fold lift), while Gradient boosting has the higher AUC. The per-fold lift gap (1.59 vs 1.47) is smaller than the spread across folds, so the two models should be treated as about equal. The data does not show the more complex model earning its complexity.
- **Results vary by fold:** the spread in the per-fold column is large because fold base rates differ widely (section 2).
- **The floor matters:** without the 100-impression floor, Gradient boosting looked far better (P@10 1.00, P@500 0.866), but its top 100 pages had a median of 1 impression, and 97% had 5 or fewer, where the label is forced. These numbers are not used as results.
- **Still to test in section 4:** the 90-day totals overlap the label windows, so part of this lift may be declines that are already visible. Section 4 checks how much.

## 4. Errors and interpretation

Three checks on what the models lean on and where they go wrong:

1. **What they lean on:** permutation importance for Gradient boosting (how much AUC drops when one feature is shuffled, measured on held-out folds), and the numeric coefficients of Logistic Regression.
2. **How much is overlap:** the 90-day activity numbers include the 30-day windows that define the label. Two tests: how many top-500 pages already have zero impressions in the last 30 days (a certain decline), and an ablation that keeps only page attributes and drops every number built from 90-day activity. The 30-day columns are used here as a diagnostic only, never as model inputs.
3. **What the errors look like:** for the model with the higher P@500, the pages in its top 500 that did not decay compared with those that did, and precision by volume bucket.

In [15]:
from sklearn.inspection import permutation_importance

imps = []
for (train_idx, test_idx), gb in zip(FOLDS, boosting_models):
    r = permutation_importance(gb, X.iloc[test_idx], y[test_idx], scoring="roc_auc",
                               n_repeats=3, random_state=SEED)
    imps.append(r.importances_mean)
gb_importance = pd.Series(np.mean(imps, axis=0), index=FEATURES).sort_values(ascending=False)
print("Gradient boosting, permutation importance on held-out folds (drop in AUC), top 8:")
print(gb_importance.head(8).round(4).to_string())

# Logistic Regression: standardized coefficients of the numeric features
# (fit on all pages, for reading only; the scores above come from the held-out folds)
lr_all = make_logistic().fit(X_lr, y)
coef = pd.Series(lr_all.named_steps["model"].coef_[0][:len(NUMERIC)], index=NUMERIC).sort_values()
print("\nLogistic Regression, numeric coefficients (per standard deviation):")
print("  most negative:", coef.head(5).round(2).to_dict())
print("  most positive:", coef.tail(5).round(2).to_dict())

Gradient boosting, permutation importance on held-out folds (drop in AUC), top 8:
avg_position             0.0397
days_with_impressions    0.0321
impressions_90d          0.0281
content_age_days         0.0280
ctr                      0.0247
clicks_90d               0.0241
scroll_rate              0.0104
days_with_sessions       0.0041

Logistic Regression, numeric coefficients (per standard deviation):
  most negative: {'users_90d': -0.94, 'avg_position': -0.42, 'days_with_sessions': -0.4, 'days_with_impressions': -0.31, 'ctr': -0.26}
  most positive: {'pageviews_90d': 0.14, 'scroll_rate': 0.14, 'scroll_events_90d': 0.27, 'word_count': 0.35, 'sessions_90d': 0.74}


In [16]:
last30_zero = (df["impressions_last_30d"] == 0).values    # diagnostic only, never a feature
print(f"Pages with zero impressions in the last 30 days: {last30_zero.mean():.1%} of the review population")
for name in ["Logistic Regression", "Gradient boosting"]:
    top = np.lexsort((ids, -oof[name]))[:500]
    print(f"  {name}: {last30_zero[top].mean():.1%} of its top 500 have zero recent impressions; "
          f"precision on the rest: {y[top][~last30_zero[top]].mean():.3f}")

# Ablation: page attributes only (drop every number built from 90-day activity)
ACTIVITY = [c for c in ["impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
                        "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
                        "days_with_impressions", "days_with_sessions", "ctr", "avg_position",
                        "engagement_rate", "scroll_rate", "ai_traffic_pct",
                        "impression_tier", "position_tier"] if c in FEATURES]
ATTRIBUTES = [c for c in FEATURES if c not in ACTIVITY]

def make_logistic_on(cols):
    num = [c for c in cols if c in NUMERIC]
    cat = [c for c in cols if c in CATEGORICAL]
    prep = ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), num),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="missing")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat)])
    return Pipeline([("prep", prep), ("model", LogisticRegression(max_iter=1000, random_state=SEED))])

oof_attr = {"Logistic Regression": np.zeros(len(y)), "Gradient boosting": np.zeros(len(y))}
for train_idx, test_idx in FOLDS:
    lr = make_logistic_on(ATTRIBUTES).fit(X_lr.iloc[train_idx][ATTRIBUTES], y[train_idx])
    oof_attr["Logistic Regression"][test_idx] = lr.predict_proba(X_lr.iloc[test_idx][ATTRIBUTES])[:, 1]
    gb = make_boosting().fit(X.iloc[train_idx][ATTRIBUTES], y[train_idx])
    oof_attr["Gradient boosting"][test_idx] = gb.predict_proba(X.iloc[test_idx][ATTRIBUTES])[:, 1]

abl = pd.DataFrame([{"model": name, "features": f"all {len(FEATURES)}",
                     "P@500": p_at_k(y, oof[name], 500, ids), "AUC": roc_auc_score(y, oof[name])} for name in oof] +
                   [{"model": name, "features": f"attributes only ({len(ATTRIBUTES)})",
                     "P@500": p_at_k(y, oof_attr[name], 500, ids), "AUC": roc_auc_score(y, oof_attr[name])} for name in oof_attr]).round(3)
print("\nAblation: page attributes only (no 90-day activity numbers)")
print(abl.to_string(index=False))

Pages with zero impressions in the last 30 days: 1.1% of the review population
  Logistic Regression: 12.2% of its top 500 have zero recent impressions; precision on the rest: 0.768
  Gradient boosting: 5.2% of its top 500 have zero recent impressions; precision on the rest: 0.694

Ablation: page attributes only (no 90-day activity numbers)
              model             features  P@500   AUC
Logistic Regression               all 34  0.796 0.638
  Gradient boosting               all 34  0.710 0.667
Logistic Regression attributes only (17)  0.580 0.565
  Gradient boosting attributes only (17)  0.484 0.592


In [17]:
best = max(oof, key=lambda n: p_at_k(y, oof[n], 500, ids))
top = np.lexsort((ids, -oof[best]))[:500]
hit = y[top] == 1
cols = ["impressions_90d", "days_with_impressions", "avg_position", "ctr", "days_since_last_update"]
print(f"Top 500 of {best}: {hit.sum()} decayed (hits), {(~hit).sum()} not decayed (misses)")
print(pd.DataFrame({"hits": df.loc[top[hit], cols].median(),
                    "misses": df.loc[top[~hit], cols].median(),
                    "all pages": df[cols].median()}).round(2).to_string())

bucket = pd.cut(df["impressions_90d"], [100, 500, 2000, 10000, np.inf],
                labels=["100-499", "500-1,999", "2,000-9,999", "10,000+"], right=False)
by_bucket = pd.DataFrame({"in_top500": pd.Series(bucket.iloc[top].values).value_counts(),
                          "precision_in_top500": pd.Series(y[top]).groupby(bucket.iloc[top].values, observed=True).mean(),
                          "decay_rate_all_pages": df.groupby(bucket, observed=True)["is_decayed"].mean()}).round(3)
print("\nBy volume bucket:")
print(by_bucket.to_string())

Top 500 of Logistic Regression: 398 decayed (hits), 102 not decayed (misses)
                         hits  misses  all pages
impressions_90d         357.0   368.0    1728.00
days_with_impressions    57.5    58.0      88.00
avg_position              6.0     7.2      12.20
ctr                       0.0     0.0       0.15
days_since_last_update   20.0    20.0      22.00

By volume bucket:
             in_top500  precision_in_top500  decay_rate_all_pages
100-499            308                0.802                 0.558
500-1,999          130                0.838                 0.536
2,000-9,999         58                0.672                 0.513
10,000+              4                0.750                 0.412


**Observed**

**What the models lean on.** Gradient boosting's most important features (AUC drop when shuffled on held-out folds) are `avg_position` (0.040), `days_with_impressions` (0.032), `impressions_90d` (0.028), `content_age_days` (0.028), `ctr` (0.025) and `clicks_90d` (0.024); all but `content_age_days` are built from 90-day activity. In Logistic Regression, `users_90d` (-0.94) and `sessions_90d` (+0.74) have opposite signs, which usually means correlated features sharing one effect, so single coefficients are not read as separate effects.

**How much is overlap.** Only 1.1% of pages have zero impressions in the last 30 days, but 12.2% of Logistic Regression's top 500 and 5.2% of Gradient boosting's do; those are certain declines. Without them, precision is 0.768 and 0.694 (base rate 0.514). With page attributes only, the models fall to AUC 0.565 and 0.592 and P@500 0.580 and 0.484 (the latter below the base rate), against 0.796 and 0.710 with all features. So most of the lift comes from 90-day activity numbers that overlap the label windows: part of what the models find is a decline that has already started, visible as fewer active days (median about 58 versus 88 for all pages) and lower volume.

**What the errors look like.** Of Logistic Regression's top 500, 398 decayed and 102 did not. The misses look almost the same as the hits on the columns checked (median impressions 368 vs 357, days with impressions 58 vs 57.5, days since update 20 vs 20); the only visible gap is average position (7.2 vs 6.0). Precision is above the overall decay rate in the 100-499 bucket (0.802 vs 0.558, n = 308), the 500-1,999 bucket (0.838 vs 0.536, n = 130) and the 2,000-9,999 bucket (0.672 vs 0.513, n = 58); the 10,000+ bucket has only 4 pages. About 62% of the top 500 sit in the lowest-volume bucket.

**Complexity.** Logistic Regression matches or beats Gradient boosting at the top of the list. Their per-fold AUC (0.650 vs 0.654) and per-fold lift (1.59 vs 1.47) differ by less than the spread across folds, so the more complex model did not earn its complexity here.

**Model vs baseline (21,758 pages, base rate 0.514)**

| Method | P@500 | Lift@500 | AUC | Per-fold lift, top 2% |
|---|---|---|---|---|
| Baseline (Week-4 rule) | 0.618 | 1.20x | 0.511 | 1.13 ± 0.11 |
| Logistic Regression | 0.796 | 1.55x | 0.638 | 1.59 ± 0.26 |
| Gradient boosting | 0.710 | 1.38x | 0.667 | 1.47 ± 0.32 |

**Reading this honestly:** both models beat the rule at the top of the list, so they are useful for prioritizing pages whose recent activity is already weak. This is directional evidence from a single snapshot: not a forecast of future decay and not a causal result.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.